<a href="https://colab.research.google.com/github/igoldin670/BlackJack-RL/blob/main/BlackJack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Blackjack Simulator

This notebook implements a simple Blackjack simulator that can later be used for Q-learning.

The player starts with two cards. The player can either:

- `"hit"`: draw another card
- `"stand"`: stop and let the dealer play

The game returns one of four outcomes:

- `"in_play"`: the game continues
- `"bust"`: the player went over 21
- `"win"`: the player wins
- `"lose"`: the player loses

In [70]:
import random

## Create and Shuffle the Deck

We use a global deck variable as the source of randomness for the simulator.

Face cards are represented as `10`.

Aces are represented as `"A"` because they can count as either 1 or 11.

In [71]:
deck = []


def create_deck():
    return [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, "A"] * 4


def shuffle_deck():
    global deck
    deck = create_deck()
    random.shuffle(deck)


def draw_card():
    global deck

    if len(deck) == 0:
        shuffle_deck()

    return deck.pop()

## Update a Blackjack Hand

The state of a hand is represented as:

```python
(total, usable_ace)
```

`total` is the current value of the hand.

`usable_ace` is True if one ace is currently being counted as 11.

In [72]:
def add_card_to_state(total, usable_ace, card):
    if card == "A":
      if total + 11 <= 21:
        total += 11
        usable_ace = True
      else:
        total += 1
    else:
      total += card

    if total > 21 and usable_ace:
        total -= 10
        usable_ace = False

    return total, usable_ace

## Set Up a New Game

The `setup_game()` function starts a new game.

It:

1. Shuffles a new deck.
2. Deals two cards to the player.
3. Returns the player's initial state.

The state is:

```python
(player_total, usable_ace)
```

In [73]:
def setup_game():
    shuffle_deck()

    player_total = 0
    usable_ace = False

    for _ in range(2):
        card = draw_card()
        player_total, usable_ace = add_card_to_state(player_total, usable_ace, card)

    return player_total, usable_ace

## Dealer Turn

The dealer gets two cards.

Then the dealer keeps hitting until their hand value is at least 17.

If the dealer goes over 21, the dealer busts.

In [74]:
def dealer_play():
    dealer_total = 0
    usable_ace = False

    for _ in range(2):
        card = draw_card()
        dealer_total, usable_ace = add_card_to_state(dealer_total, usable_ace, card)

    while dealer_total < 17:
        card = draw_card()
        dealer_total, usable_ace = add_card_to_state(dealer_total, usable_ace, card)

    return dealer_total

## Transition Function

The `step()` function is the main transition function.

It takes:

```python
state
action
```

where:

```python
state = (player_total, usable_ace)
```

and action is either:

`"hit"`
or
`"stand"`

The function returns:

next_state, outcome

The outcome can be:

`"in_play"`
`"bust"`
`"win"`
`"lose"`

In [75]:
def step(state, action):
    player_total, usable_ace = state

    if action == "hit":
        card = draw_card()
        player_total, usable_ace = add_card_to_state(player_total, usable_ace, card)

        if player_total > 21:
            return None, "bust"

        return (player_total, usable_ace), "in_play"

    elif action == "stand":
        dealer_total = dealer_play()

        if dealer_total > 21:
            return state, "win"
        elif player_total > dealer_total:
            return state, "win"
        else:
            return state, "lose"

    else:
        raise ValueError("Invalid action: {}".format(action))

## Test One Game

This cell tests one simple game.

The player starts with two cards, hits once, and if the game is still going, stands.

In [76]:
state = setup_game()
print("Initial player state:", state)

state, outcome = step(state, "hit")
print("After hit:", state, outcome)

if outcome == "in_play":
    state, outcome = step(state, "stand")
    print("After stand:", state, outcome)

Initial player state: (11, False)
After hit: (12, False) in_play
After stand: (12, False) lose


## Q-Learning Format

For Q-learning, it is often better to return:

```python
next_state, reward, done
```

where:

- win gives reward `+1`
- lose gives reward `-1`
- bust gives reward `-1`
- in play gives reward `0`

The `done` variable is `True` when the game is over.

In [77]:
def step_q_learning(state, action):
    next_state, outcome = step(state, action)

    if outcome == "win":
        reward = 1
        done = True
    elif outcome == "lose":
        reward = -1
        done = True
    elif outcome == "bust":
        reward = -1
        done = True
    else:
        reward = 0
        done = False

    return next_state, reward, done

In [78]:
state = setup_game()
print("Initial state:", state)

next_state, reward, done = step_q_learning(state, "hit")

print("Next state:", next_state)
print("Reward:", reward)
print("Done:", done)

Initial state: (8, False)
Next state: (19, True)
Reward: 0
Done: False


# Q-Learning for Simple Blackjack

Now we build a Q-learner for the Blackjack simulator.

The Q-table stores the value of each action in each state.

A state looks like:

```python
(player_total, usable_ace)
```

An action is either:

`"hit"`
or
`"stand"`

So one Q-table entry looks like:

```python
Q[(18, False)]["hit"]
Q[(18, False)]["stand"]
```

The value estimates how good that action is from that state.

## Set Up the Q-Table

We use a dictionary for the Q-table.

Each state has two possible actions:

- `"hit"`
- `"stand"`

At the beginning, all Q-values are set to `0.0`.

In [79]:
import numpy as np
import pandas as pd

actions = ["hit", "stand"]

player_totals = range(4, 22)
usable_ace_values = [False, True]

state_index = pd.MultiIndex.from_product(
    [player_totals, usable_ace_values],
    names=["player_total", "usable_ace"]
)

Q = pd.DataFrame(
    np.zeros((len(state_index), len(actions))),
    index=state_index,
    columns=actions
)

Q.head(10)

hit  stand
player_total usable_ace            
4            False       0.0    0.0
             True        0.0    0.0
5            False       0.0    0.0
             True        0.0    0.0
6            False       0.0    0.0
             True        0.0    0.0
7            False       0.0    0.0
             True        0.0    0.0
8            False       0.0    0.0
             True        0.0    0.0

## Q-Learning Update Rule

The Q-learning formula is:

```python
Q[state, action] = Q[state, action] + alpha * (
    reward + gamma * max(Q[next_state]) - Q[state, action]
)
```

For this assignment:

`gamma = 1`

because games end quickly, so we do not discount future rewards.


In [80]:
def update_q_value(Q, state, action, reward, next_state, done, alpha = 0.1, gamma = 1):
    current_q = Q.loc[state, action]

    if done:
      target = reward

    else:
        best_next_q = Q.loc[next_state].max()
        target = reward + gamma * best_next_q

    Q.loc[state, action] = current_q + alpha * (target - current_q)

## Exploration Function

This function plays many blackjack games.

The player chooses actions randomly.

After each action, the Q-table is updated.

In [81]:
def explore_blackjack(Q, num_games = 10000, alpha = 0.1, gamma = 1):
    results = {
        "win":0,
        "lose":0,
        "bust":0
    }

    for game in range(num_games):
        state = setup_game()
        done = False

        while not done:
            action = random.choice(actions)
            next_state, reward, done = step_q_learning(state, action)
            update_q_value(Q, state, action, reward, next_state, done, alpha, gamma)

            if done:
              if reward == 1:
                results["win"] += 1
              elif next_state is None:
                results["bust"] += 1
              else:
                results["lose"] += 1
            else:
              state = next_state

    return Q, results

## Run Q-Learning

Now we run the exploration function for 10,000 games.

In [82]:
Q, results = explore_blackjack(
    Q=Q,
    num_games=10000,
    alpha=0.1,
    gamma=1
)

results

{'win': 2910, 'lose': 3898, 'bust': 3192}

In [83]:
Q

hit     stand
player_total usable_ace                    
4            False      -0.094192 -0.285525
             True        0.000000  0.000000
5            False      -0.256007 -0.488579
             True        0.000000  0.000000
6            False      -0.154053 -0.432922
             True        0.000000  0.000000
7            False      -0.224736 -0.498876
             True        0.000000  0.000000
8            False      -0.101283 -0.112741
             True        0.000000  0.000000
9            False      -0.052847 -0.524932
             True        0.000000  0.000000
10           False       0.013459 -0.496480
             True        0.000000  0.000000
11           False       0.213467 -0.513874
             True        0.000000  0.000000
12           False      -0.414687 -0.660897
             True       -0.154801 -0.745813
13           False      -0.446459 -0.324630
             True       -0.020558 -0.765871
14           False      -0.512098 -0.799382
             True       -0.190228 -0.812942
15           False      -0.774521 -0.364619
             True       -0.128784 -0.448960
16           False      -0.211635 -0.489824
             True       -0.085680 -0.265916
17           False      -0.686747 -0.180776
             True       -0.180851 -0.034945
18           False      -0.600122 -0.394794
             True       -0.147593 -0.042399
19           False      -0.768781  0.289263
             True        0.001972  0.214017
20           False      -0.717741  0.607042
             True       -0.003733  0.766396
21           False      -1.000000  0.672458
             True        0.107887  0.905275

## Best Learned Action for Each State

This shows whether the learned policy prefers `"hit"` or `"stand"` for each state.

In [84]:
policy = Q.idxmax(axis=1).to_frame(name="best_action")

policy.head(20)

best_action
player_total usable_ace            
4            False              hit
             True               hit
5            False              hit
             True               hit
6            False              hit
             True               hit
7            False              hit
             True               hit
8            False              hit
             True               hit
9            False              hit
             True               hit
10           False              hit
             True               hit
11           False              hit
             True               hit
12           False              hit
             True               hit
13           False            stand
             True               hit

In [85]:
policy

best_action
player_total usable_ace            
4            False              hit
             True               hit
5            False              hit
             True               hit
6            False              hit
             True               hit
7            False              hit
             True               hit
8            False              hit
             True               hit
9            False              hit
             True               hit
10           False              hit
             True               hit
11           False              hit
             True               hit
12           False              hit
             True               hit
13           False            stand
             True               hit
14           False              hit
             True               hit
15           False            stand
             True               hit
16           False              hit
             True               hit
17           False            stand
             True             stand
18           False            stand
             True             stand
19           False            stand
             True             stand
20           False            stand
             True             stand
21           False            stand
             True             stand

## Combine Q-Values and Best Action

This table shows the Q-value for hitting, the Q-value for standing, and the best action.

In [86]:
Q_with_policy = Q.copy()
Q_with_policy["best_action"] = Q.idxmax(axis=1)

Q_with_policy

hit     stand best_action
player_total usable_ace                                
4            False      -0.094192 -0.285525         hit
             True        0.000000  0.000000         hit
5            False      -0.256007 -0.488579         hit
             True        0.000000  0.000000         hit
6            False      -0.154053 -0.432922         hit
             True        0.000000  0.000000         hit
7            False      -0.224736 -0.498876         hit
             True        0.000000  0.000000         hit
8            False      -0.101283 -0.112741         hit
             True        0.000000  0.000000         hit
9            False      -0.052847 -0.524932         hit
             True        0.000000  0.000000         hit
10           False       0.013459 -0.496480         hit
             True        0.000000  0.000000         hit
11           False       0.213467 -0.513874         hit
             True        0.000000  0.000000         hit
12           False      -0.414687 -0.660897         hit
             True       -0.154801 -0.745813         hit
13           False      -0.446459 -0.324630       stand
             True       -0.020558 -0.765871         hit
14           False      -0.512098 -0.799382         hit
             True       -0.190228 -0.812942         hit
15           False      -0.774521 -0.364619       stand
             True       -0.128784 -0.448960         hit
16           False      -0.211635 -0.489824         hit
             True       -0.085680 -0.265916         hit
17           False      -0.686747 -0.180776       stand
             True       -0.180851 -0.034945       stand
18           False      -0.600122 -0.394794       stand
             True       -0.147593 -0.042399       stand
19           False      -0.768781  0.289263       stand
             True        0.001972  0.214017       stand
20           False      -0.717741  0.607042       stand
             True       -0.003733  0.766396       stand
21           False      -1.000000  0.672458       stand
             True        0.107887  0.905275       stand